# PCA: Simple OHCO

This notebook follows the M07 PCA example using the reduced and L2-normalized TFIDF table created from the simple-OHCO constitution corpus.

# Set Up

## Import

In [24]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
from sklearn.decomposition import PCA

pio.renderers.default = "notebook_connected"

## Config

In [25]:
data_home = "."
output_dir = "output"
data_prefix = "constitutions-simple-ohco"
source_dir = output_dir
CSV_DELIM = "|"

bag = "CONSTITUTIONS"
n_components = 5

# Prepare the Data

## Import Tables

In [26]:
LIB = pd.read_csv(f"{source_dir}/{data_prefix}-LIB.csv", sep=CSV_DELIM).set_index("constitution_id")
DOC = pd.read_csv(f"{source_dir}/{data_prefix}-DOC-{bag}.csv", sep=CSV_DELIM).set_index("constitution_id")
SIGS = pd.read_csv(f"{source_dir}/{data_prefix}-SIGS-{bag}.csv", sep=CSV_DELIM).set_index("term_str")
TFIDF_L2 = pd.read_csv(f"{source_dir}/{data_prefix}-TFIDF_L2-{bag}.csv", sep=CSV_DELIM).set_index("constitution_id")

In [27]:
TFIDF_L2.shape

(192, 2410)

In [28]:
DOC = DOC.join(LIB[["country", "file_year", "original_year", "revision_year", "title"]], rsuffix="_lib")
DOC.head()

,n_tokens,n_types,pkr,country,file_year,original_year,revision_year,title,country_lib,file_year_lib,original_year_lib,revision_year_lib,title_lib
constitution_id,,,,,,,,,,,,,
Afghanistan_2004,10435,1618,0.155055,Afghanistan,2004,2004.0,NaN,Afghanistan,Afghanistan,2004,2004.0,NaN,Afghanistan
Albania_2008,13896,1748,0.125792,Albania,2008,1998.0,2008.0,Albania,Albania,2008,1998.0,2008.0,Albania
Algeria_2008,10605,1662,0.156719,Algeria,2008,1963.0,2008.0,Algeria,Algeria,2008,1963.0,2008.0,Algeria
Andorra_1993,8722,1484,0.170144,Andorra,1993,1993.0,NaN,Andorra,Andorra,1993,1993.0,NaN,Andorra
Angola_2010,26657,2750,0.103162,Angola,2010,2010.0,NaN,Angola,Angola,2010,2010.0,NaN,Angola


# Run PCA

In [29]:
pca_engine = PCA(n_components=n_components)
DCM = pd.DataFrame(pca_engine.fit_transform(TFIDF_L2), index=TFIDF_L2.index)
DCM.columns = ["PC{}".format(i) for i in DCM.columns]
DCM.head()

,PC0,PC1,PC2,PC3,PC4
constitution_id,,,,,
Afghanistan_2004,-0.000069,-0.000198,-0.000229,-0.000228,0.000983
Albania_2008,0.000389,-0.000481,0.000060,-0.000410,-0.000036
Algeria_2008,0.000420,-0.000521,0.000058,-0.000285,0.000027
Andorra_1993,0.000368,-0.000423,0.000121,-0.000376,-0.000067
Angola_2010,0.000365,-0.000438,0.000101,-0.000426,-0.000101


In [30]:
COMPS = pd.DataFrame(pca_engine.components_.T, index=TFIDF_L2.columns)
COMPS.columns = ["PC{}".format(i) for i in COMPS.columns]
COMPS.index.name = "term_str"
COMPS.head()

,PC0,PC1,PC2,PC3,PC4
term_str,,,,,
honor,0.006179,-0.008964,0.000288,-0.002084,0.006906
labor,0.015963,-0.011075,0.001991,0.003819,-0.000800
occasion,-0.009619,0.008226,-0.003863,0.001310,-0.000183
stage,-0.001527,0.001261,-0.000339,0.000712,0.001195
integral,0.003621,-0.005613,0.000009,-0.002698,0.003631


In [31]:
PCA_VAR = pd.DataFrame({
    "eigenvalue": pca_engine.explained_variance_,
    "explained_variance_ratio": pca_engine.explained_variance_ratio_,
}, index=DCM.columns)
PCA_VAR.index.name = "component"
PCA_VAR["cum_explained_variance_ratio"] = PCA_VAR.explained_variance_ratio.cumsum()
PCA_VAR

,eigenvalue,explained_variance_ratio,cum_explained_variance_ratio
component,,,
PC0,1.390096e-06,0.129525,0.129525
PC1,1.189572e-06,0.110841,0.240367
PC2,9.068917e-07,0.084502,0.324868
PC3,7.825862e-07,0.072919,0.397788
PC4,4.710945e-07,0.043895,0.441683


# Inspect Components

In [32]:
def top_terms_for_pc(pc_num, n=10):
    pc = f"PC{pc_num}"
    positive = COMPS[pc].sort_values(ascending=False).head(n).rename("positive")
    negative = COMPS[pc].sort_values().head(n).rename("negative")
    return pd.concat([positive.reset_index(), negative.reset_index()], axis=1)


top_terms_for_pc(0)

,term_str,positive,term_str,negative
0,art,0.564763,subsection,-0.517002
1,congress,0.118764,governor,-0.276940
2,chamber,0.063881,house,-0.258678
3,deputies,0.043723,ad,-0.142898
4,down,0.032722,speaker,-0.132396
5,communal,0.030399,senate,-0.103693
6,autonomous,0.029426,advice,-0.100472
7,peoples,0.027184,officer,-0.079939
8,municipalities,0.026114,references,-0.075176
9,statute,0.025318,director,-0.075099


In [33]:
top_terms_for_pc(1)

,term_str,positive,term_str,negative
0,art,0.798058,congress,-0.202999
1,subsection,0.347118,peoples,-0.049757
2,governor,0.195440,chamber,-0.043297
3,house,0.173622,deputies,-0.042989
4,ad,0.105201,organic,-0.034283
5,speaker,0.087471,autonomous,-0.032645
6,advice,0.067082,everyone,-0.031210
7,references,0.054141,statute,-0.030065
8,crown,0.053975,chambers,-0.025659
9,senate,0.049024,organs,-0.024345


In [34]:
COMPS.PC0.sort_values(ascending=False).head(5)

term_str
art         0.564763
congress    0.118764
chamber     0.063881
deputies    0.043723
down        0.032722
Name: PC0, dtype: float64

In [35]:
COMPS.PC1.sort_values().head(5)

term_str
congress   -0.202999
peoples    -0.049757
chamber    -0.043297
deputies   -0.042989
organic    -0.034283
Name: PC1, dtype: float64

# Visualize

In [36]:
def vis_pcs(a, b, col="country"):
    fig = px.scatter(
        DCM,
        f"PC{a}",
        f"PC{b}",
        color=DOC[col],
        hover_name=DCM.index,
        hover_data={
            "title": DOC.title,
            "file_year": DOC.file_year,
            "n_tokens": DOC.n_tokens,
        },
        marginal_x="box",
        height=800,
        width=1000,
        template="plotly_white",
    )
    fig.update_traces(marker={"size": 9, "opacity": 0.8, "line": {"width": 0.5, "color": "#333333"}})
    fig.update_layout(legend_title_text=col, xaxis_title=f"PC{a}", yaxis_title=f"PC{b}")
    return fig

In [37]:
vis_pcs(0, 1)

In [38]:
vis_pcs(2, 3)

## By Feature

In [39]:
def plot_by_feature(a, b, col="file_year"):
    x = f"PC{a}"
    y = f"PC{b}"
    X = DCM.groupby(DOC[col]).mean()
    fig = px.scatter(
        X.reset_index(),
        x=x,
        y=y,
        size=DOC.groupby(col).n_tokens.sum(),
        height=500,
        width=700,
        text=col,
        template="plotly_white",
    )
    fig.update_traces(marker={"opacity": 0.8, "line": {"width": 0.5, "color": "#333333"}})
    fig.update_layout(xaxis_title=x, yaxis_title=y)
    return fig

In [40]:
plot_by_feature(0, 1, "file_year")

In [41]:
plot_by_feature(2, 3, "file_year")

# Save

The export lines are intentionally commented out so the PCA tables can be inspected before saving.

In [42]:
save_path = f"{output_dir}/{data_prefix}"

# DCM.to_csv(f"{save_path}-PCA_DCM-{bag}.csv", sep=CSV_DELIM)
# COMPS.to_csv(f"{save_path}-PCA_COMPS-{bag}.csv", sep=CSV_DELIM)
# PCA_VAR.to_csv(f"{save_path}-PCA_VAR-{bag}.csv", sep=CSV_DELIM)